In [1]:
# was dedup-dependency
#from pathlib import Path
#from PIL import Image
#from hashlib import sha1
#from glob import iglob

# DEDUP

> Dedup was done successfully,  
> but due to Jupyter Notebook crash and save error,  
> All code with 3-hours of effors was gone.  
>  
> 1. Tried every method to recover original code,  
> but the last saved checkpoint was 3-hours ago.  
> 2. Tried to search Chrome browser cache binary,  
> browser cache was already removed.  
>  
> The result `index.txt` file is made successfully,  
> here, passing full-reimplement of indexing code.  

```
# Will consume some memory
class LocalIndex:
    def __init__(self):
        class CaptchaString: pass
        class FileHash: pass
        class FilePath: pass

        # Docs
        self._index: dict[CaptchaString: list[tuple[FilePath, FileHash]]] = dict()

    # This is temporal class,
    # Not optimized and structurized.
    def push(self, filepath:str) -> bool:

        # Will use checksum always.
        # Calculate here first.
        with open(filepath, "rb") as rawfile:
            file_hash_hex = sha1(rawfile, usedforsecurity=False).hexdigest()

        # Check file in index
        basename_wo_ext = Path(filepath).stem
        if exists := self._index.get(basename_wo_ext, None):
            for exist in exists:
                # Compare exist hash of file with same filename
                if (file_hash_hex == exist[1]):
                    return False;
            self._index[basename_wo_ext].append(
                (filepath, file_hash_hex)
            )
        else:
            self._index[basename_wo_ext] = [(filepath, file_hash_hex), ]

        return True;

local_index = LocalIndex()
# All non-PNG files were converted into PNG
for filepath in iglob("data/**/*.png"):
    if not local_index.push(filepath):
        print(filepath)
```


> According to previous experiment,  
> the biggest image size was (256x256).
>  
> And there are some grayscale and RGBA images,  
> here, converting every source image into  
> `(256, 256, 3)` sized images

# Resize to unify shape and format

In [4]:
import torchvision.transforms as T
import numpy as np
import os, shutil, time
from PIL import Image
from collections import Counter
from pathlib import Path

In [5]:
def get_dominant_corner_color(img: Image, _sample=0.05):
    img = np.array(img)
    w, h = img.shape[:2]
    channel = 1 if img.ndim == 2 else img.shape[2]

    pw = max(1, int(w * _sample))
    ph = max(1, int(h * _sample))

    corners = list()
    corners.append(img[0:ph, 0:pw]) # TL
    corners.append(img[0:ph, -pw:]) # TB
    corners.append(img[-ph:, 0:pw]) # BL
    corners.append(img[-ph:, -pw:]) # BR

    corners = np.concatenate([corner.reshape(-1, channel) for corner in corners], axis=0)

    pixels = [tuple(rgb) for rgb in corners]
    most_common = Counter(pixels).most_common(1)[0][0]

    return most_common;


In [6]:
def rgb_from_grayscale(img) -> Image:
    return img.convert("RGB")

In [7]:
def rgb_from_rgba(img) -> Image:
    bg_color = get_dominant_corner_color(img, _sample=0.1)
    bg = Image.new("RGB", img.size, bg_color)
    return Image.alpha_composite(bg, img).convert('RGB')

In [8]:
# max among `{ (40, 150, 3), (50, 200, 3), (50, 180), (50, 200, 4), (256, 256, 3) }`
MAX_W, MAX_H = (256, 256)

def reshape(img: Image) -> Image:
    hw = (MAX_W-img.width )//2
    hh = (MAX_H-img.height)//2

    unify = T.Compose([
        T.Pad(
            (hw, hh, hw, hh),
            fill=get_dominant_corner_color(img, _sample=0.1),
        ),
        T.Resize(
            (256,256)
        ),
    ])

    return unify(img);

In [9]:
_dst = "./data/ready/"
shutil.rmtree(_dst)
os.makedirs(_dst, exist_ok=False)
with open("./data/index.txt", "r", encoding="utf-8") as index_txt:
    _i = 0;
    _proc_per = 1000;
    _last_time = time.perf_counter()
    for filepath in index_txt.readlines():
        _i += 1;
        filepath = filepath.strip()
        basename_wo_ext = Path(filepath).stem

        if not _i % _proc_per:
            _time = time.perf_counter()
            _dt = _time - _last_time
            _last_time = _time
            print(f"Processed `{_i}` [{filepath}] ({_dt:.02f}s)", end="                \r")

        # Verify valid
        try:
            Image.open(filepath).verify()
        except Exception as _:
            print(f"\nbroken: `{filepath}`)")
            Path(filepath).unlink(missing_ok=True)
            continue;

        # Verify consume Image
        img = Image.open(filepath)

        # Grayscale
        if len(img.size)==2 or img.size[2]==1:
            img = rgb_from_grayscale(img)
        # RGBA
        elif img.size[2]==4:
            img = rbg_from_rgba(img)
        # Something went wrong
        else:
            print(f"\n\nError: Non-Grayscale, Non-RGBA image: {filepath}\n\n")
            raise Exception("panic!();")

        img = reshape(img)

        # if there are more than 999 image for one captcha combination, 
        # this will result in overwrite the 100th result, 
        # but this will be an extreamly rare case. 
        for i in range(999):
            save_filepath = f"{_dst}{basename_wo_ext}.{i}.png"
            if Path(save_filepath).is_file():
                continue;
            break;
        img.save(save_filepath)
    print(f"Processed `{_i}` [{filepath}]", end="                \r")

broken: rm `data\archive (1)\Large_Captcha_Dataset\4q2wA.png`)X.png]                
Processed `198461` [data\archive (3)\data\val\nn6w6.jpg]                            